In [1]:
# P(c3 | c1, c2)

In [54]:
words = open('names.txt', 'r').read().splitlines()

In [55]:
import torch
import torch.nn.functional as F

In [56]:
alphabets = ['.'] + sorted(list(set(''.join(words))))

stoi = {s: i for i, s in enumerate(alphabets)}
itos = {i: s for s, i in stoi.items()}
btoi = {}
n = 0
for ch1 in alphabets:
  for ch2 in alphabets:
    btoi[f'{ch1}{ch2}'] = n
    n += 1
itob = {i:s for s, i in btoi.items()}




In [74]:
# create data set

xs, ys = [], []

for w_idx in range(len(words)):
  current_word_chars = ['.','.'] + list(words[w_idx]) + ['.'] # adding extra '.' at the start such that ix at sampleing which is by default 0 -> '..' should now where to point at from the training
  for i in range(len(current_word_chars) - 2):
    bigram = f'{current_word_chars[i]}{current_word_chars[i+1]}'
    ix = btoi[bigram]
    iy = stoi[current_word_chars[i+2]]

    xs.append(ix)
    ys.append(iy)

    # print(f'input: {bigram}; output: {current_word_chars[i+2]}')

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()

# print(f'Number for examples: {num}')


# ----- NOTES ONE THE LOOP -------

# word = .atharv.
# len(word) = 8

# .a -> t (i = 0)
# at -> h (i = 1)
# th -> a (i = 2)
# ha -> r (i = 3)
# ar -> v (i = 4)
# rv -> . (i = 5)
# v. -> ? (we do not want this iteration)

# our loop should run for 6 time, hence (n = 6) = len(word) - 2

In [75]:
# initialize 'network'
g = torch.Generator().manual_seed(2147483648)
w = torch.randn((729, 27), generator=g, requires_grad=True)

In [76]:
# gradient decent
for k in range(200):

  # forward pass
  xenc = F.one_hot(xs, num_classes=729).float()
  logits = (xenc @ w) # log-counts
  counts = logits.exp() # equivalent to N in the previous model
  probs = counts / counts.sum(1, keepdim=True) # normalizing each row
  loss = -(probs[torch.arange(num), ys]).log().mean() + 0.1*(w**2).mean()
  print(loss.item())

  # backward pass
  w.grad = None
  loss.backward()

  # update
  w.data += -50 * w.grad

3.780505657196045
3.6921989917755127
3.6240153312683105
3.5653769969940186
3.513382911682129
3.466278553009033
3.423083782196045
3.3832359313964844
3.34637188911438
3.3121988773345947
3.28045392036438
3.250892400741577
3.2232956886291504
3.1974737644195557
3.1732661724090576
3.150536298751831
3.1291656494140625
3.109049081802368
3.090088129043579
3.0721921920776367
3.05527400970459
3.0392537117004395
3.0240554809570312
3.009611129760742
2.9958572387695312
2.9827370643615723
2.970200777053833
2.9582016468048096
2.946699380874634
2.935657024383545
2.925041437149048
2.9148240089416504
2.9049770832061768
2.8954780101776123
2.8863039016723633
2.8774361610412598
2.8688549995422363
2.860546588897705
2.852494478225708
2.8446853160858154
2.837106704711914
2.8297476768493652
2.822596788406372
2.8156445026397705
2.8088815212249756
2.8022992610931396
2.7958900928497314
2.789646863937378
2.783562183380127
2.7776291370391846
2.7718420028686523
2.766195774078369
2.7606844902038574
2.7553024291992188


In [88]:
# sampling
for _ in range(20):
  word = []
  ix = 0
  while True:

    xenc = F.one_hot(torch.tensor([ix]), num_classes=729).float()
    logits = xenc @ w
    counts = logits.exp()
    p = counts / counts.sum(1, keepdim=True)

    yx = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()

    prev = (itob[ix])
    ix = btoi[f'{prev[1]}{itos[yx]}']

    word.append(itos[yx])
    if yx == 0:
      break

  print(''.join(word))

vitrjzppwmiyas.
son.
belvislia.
pne.
dqqpqvxzrzirestaee.
ransron.
elsopbucxfjjyjdistokftjleill.
jbher.
kany.
yah.
zxlxvjwgsongociya.
.
ke.
ccvwsynnix.
ana.
marylickus.
haaqxia.
si.
nalianasireyley.
malisxzjen.
